# Longstaff-Schwartz Method for Bermudan Options

This notebook gives the mathematical foundation behind the implementation in `longstaff_schwartz_bermudan_options_clean.ipynb`. The objective is to price a Bermudan put option using Monte Carlo simulation and least-squares regression under the risk-neutral measure.

## 1. Financial Contract

We consider a Bermudan put option written on an underlying asset $S_t$ with strike $K$ and maturity $T$.

A European put can only be exercised at maturity:

$$
\Phi(S_T) = (K - S_T)^+.
$$

An American put can be exercised at any time $t \in [0,T]$.

A Bermudan put is intermediate: exercise is allowed only on a finite grid of dates

$$
0 < t_1 < t_2 < \cdots < t_m = T.
$$

At each allowed exercise date $t_j$, the holder compares immediate exercise value

$$
\Phi(S_{t_j}) = (K - S_{t_j})^+
$$

against the value of continuing the option.

## 2. Risk-Neutral Dynamics

Under the risk-neutral measure $\mathbb{Q}$, the stock price is modeled as a geometric Brownian motion:

$$
dS_t = r S_t\,dt + \sigma S_t\,dW_t^{\mathbb{Q}},
$$

where:

- $r$ is the continuously compounded risk-free rate,
- $\sigma$ is volatility,
- $W_t^{\mathbb{Q}}$ is a Brownian motion under $\mathbb{Q}$.

The exact discretization on a time grid $0=t_0<t_1<\cdots<t_N=T$ is

$$
S_{t_{n+1}} = S_{t_n}\exp\left[\left(r - \frac{1}{2}\sigma^2\right)\Delta t + \sigma\sqrt{\Delta t}\,Z_{n+1}\right],
$$

with $Z_{n+1}\sim\mathcal{N}(0,1)$ independent across time and paths.

This is the simulation engine used in the project.

## 3. Risk-Neutral Valuation Principle

For a derivative payoff $X_T$, no-arbitrage pricing gives

$$
V_0 = \mathbb{E}^{\mathbb{Q}}\left[e^{-rT}X_T\right].
$$

For early-exercise options, the payoff time is not fixed. It is a stopping time $\tau$ chosen by the holder. The option value is therefore an optimal stopping problem:

$$
V_0 = \sup_{\tau \in \mathcal{T}} \mathbb{E}^{\mathbb{Q}}\left[e^{-r\tau}\Phi(S_\tau)\right],
$$

where $\mathcal{T}$ is the set of admissible stopping times.

For a Bermudan option, $\tau$ is restricted to the exercise grid:

$$
\tau \in \{t_1,t_2,\ldots,t_m\}.
$$

## 4. Dynamic Programming and the Snell Envelope

Define the option value on the Bermudan exercise grid by backward induction.

At maturity:

$$
V_{t_m}(S_{t_m}) = \Phi(S_{t_m}).
$$

At an earlier exercise date $t_j$, the holder chooses between exercising and continuing:

$$
V_{t_j}(S_{t_j}) = \max\left(\Phi(S_{t_j}), C_{t_j}(S_{t_j})\right),
$$

where the continuation value is

$$
C_{t_j}(S_{t_j}) = \mathbb{E}^{\mathbb{Q}}\left[e^{-r(t_{j+1}-t_j)}V_{t_{j+1}}(S_{t_{j+1}})\mid S_{t_j}\right].
$$

The process $V_{t_j}$ is the discrete-time Snell envelope of the discounted payoff process. It is the smallest supermartingale dominating the exercise payoff.

## 5. Why Regression Is Needed

In simulation, we observe many paths

$$
\left(S_{t_0}^{(i)}, S_{t_1}^{(i)}, \ldots, S_{t_N}^{(i)}\right),\quad i=1,\ldots,M.
$$

At time $t_j$, the continuation value is a conditional expectation:

$$
C_{t_j}(s) = \mathbb{E}^{\mathbb{Q}}\left[e^{-r\Delta t_j}V_{t_{j+1}}(S_{t_{j+1}})\mid S_{t_j}=s\right].
$$

This conditional expectation is not directly observable path-by-path. Longstaff-Schwartz approximates it by projecting future discounted values onto basis functions of the current state.

## 6. Least-Squares Approximation

Choose basis functions

$$
\psi_0(s),\psi_1(s),\ldots,\psi_p(s).
$$

In our implementation, we use polynomial basis functions such as

$$
\psi(s) = \left(1,s,s^2\right)
$$

for a degree-2 regression.

The continuation value is approximated by

$$
\widehat{C}_{t_j}(s) = \sum_{k=0}^{p}\beta_k\psi_k(s).
$$

The coefficients are estimated by least squares using simulated paths that are in the money:

$$
\widehat{\beta}
= \arg\min_{\beta}\sum_{i\in\mathcal{I}_j}
\left(Y_j^{(i)} - \sum_{k=0}^{p}\beta_k\psi_k(S_{t_j}^{(i)})\right)^2,
$$

where

$$
\mathcal{I}_j = \{i : \Phi(S_{t_j}^{(i)})>0\}
$$

and

$$
Y_j^{(i)} = e^{-r(t_{j+1}-t_j)}V_{t_{j+1}}^{(i)}.
$$

Restricting to in-the-money paths is standard because out-of-the-money paths have zero immediate exercise value and therefore no exercise decision to make at that date.

## 7. Exercise Rule

Once the continuation value is estimated, the optimal exercise decision is approximated by

$$
\text{exercise at } t_j
\quad\Longleftrightarrow\quad
\Phi(S_{t_j}) > \widehat{C}_{t_j}(S_{t_j}).
$$

For a put option:

$$
(K-S_{t_j})^+ > \widehat{C}_{t_j}(S_{t_j}).
$$

If exercise is optimal, the path value becomes

$$
V_{t_j}^{(i)} = \Phi(S_{t_j}^{(i)}).
$$

Otherwise, the path value is the discounted continuation value already carried backward from the next exercise date.

## 8. Backward Algorithm

The Longstaff-Schwartz algorithm proceeds backward through exercise dates.

1. Simulate $M$ risk-neutral paths of $S_t$.
2. Initialize values at maturity:

$$
V_{t_m}^{(i)} = \Phi(S_{t_m}^{(i)}).
$$

3. For $j=m-1,m-2,\ldots,1$:

$$
Y_j^{(i)} = e^{-r(t_{j+1}-t_j)}V_{t_{j+1}}^{(i)}.
$$

Estimate $\widehat{C}_{t_j}$ by least-squares regression of $Y_j^{(i)}$ against basis functions of $S_{t_j}^{(i)}$.

Then update each path by

$$
V_{t_j}^{(i)} =
\begin{cases}
\Phi(S_{t_j}^{(i)}), & \text{if } \Phi(S_{t_j}^{(i)}) > \widehat{C}_{t_j}(S_{t_j}^{(i)}),\\
Y_j^{(i)}, & \text{otherwise.}
\end{cases}
$$

4. Discount from the first exercise date back to time zero:

$$
\widehat{V}_0 = \frac{1}{M}\sum_{i=1}^{M} e^{-rt_1}V_{t_1}^{(i)}.
$$

## 9. Monte Carlo Error

The estimator is an average of discounted pathwise values:

$$
\widehat{V}_0 = \frac{1}{M}\sum_{i=1}^{M} X_i,
$$

where $X_i$ is the discounted payoff produced by the estimated stopping rule.

A standard Monte Carlo standard error estimate is

$$
\widehat{\operatorname{SE}}(\widehat{V}_0)
= \frac{s_X}{\sqrt{M}},
$$

where

$$
s_X^2 = \frac{1}{M-1}\sum_{i=1}^{M}(X_i-\bar{X})^2.
$$

This measures simulation uncertainty, but it does not fully capture regression approximation error or bias from using the same paths for regression and pricing.

## 10. European, Bermudan, and American Ordering

For a put option under the same market assumptions, the exercise rights satisfy

$$
\text{European exercise set} \subseteq \text{Bermudan exercise set} \subseteq \text{American exercise set}.
$$

Therefore, theoretically:

$$
V^{\text{European}}_0 \leq V^{\text{Bermudan}}_0 \leq V^{\text{American}}_0.
$$

Small numerical violations can happen in Monte Carlo because of finite path count, regression noise, and approximation error.

## 11. Interpretation of the Regression Plot

The diagnostic plot in the main notebook displays, for a fixed exercise date $t_j$:

- the current underlying level $S_{t_j}^{(i)}$ on the horizontal axis,
- the discounted future path value $Y_j^{(i)}$ on the vertical axis,
- the fitted continuation curve $\widehat{C}_{t_j}(s)$ in red,
- the strike $K$ as a vertical dashed line.

Mathematically, the red curve is an empirical approximation of

$$
s \mapsto \mathbb{E}^{\mathbb{Q}}\left[e^{-r(t_{j+1}-t_j)}V_{t_{j+1}}(S_{t_{j+1}})\mid S_{t_j}=s\right].
$$

The exercise boundary is implicitly determined by the crossing between immediate payoff and continuation value:

$$
K-s = \widehat{C}_{t_j}(s)
$$

for in-the-money states $s<K$.

## 12. Neural-Network Regression Extension

The polynomial Longstaff-Schwartz method approximates continuation value by a linear combination of fixed basis functions.

A neural-network continuation model replaces

$$
\widehat{C}_{t_j}(s) = \sum_{k=0}^{p}\beta_k\psi_k(s)
$$

with a nonlinear function approximator

$$
\widehat{C}_{t_j}(s) = f_{\theta_j}(s),
$$

where $f_{\theta_j}$ is trained to minimize

$$
\min_{\theta_j}\sum_{i\in\mathcal{I}_j}\left(Y_j^{(i)} - f_{\theta_j}(S_{t_j}^{(i)})\right)^2.
$$

The quant intuition is the same: approximate a conditional expectation. The difference is the function class used for the approximation.

In a one-dimensional GBM problem, polynomial regression is often sufficient. Neural regression becomes more valuable when the state vector is high-dimensional, for example with stochastic volatility, multiple assets, path-dependent features, or market state variables.

## 13. Numerical Mindset

A rigorous quant implementation should check:

- convergence as the number of Monte Carlo paths increases,
- sensitivity to the polynomial degree or model architecture,
- sensitivity to the number of exercise dates,
- consistency with European Black-Scholes prices,
- reasonable ordering between European, Bermudan, and American prices,
- stability of exercise-time distributions,
- stability of continuation-regression plots across exercise dates.

The project notebook implements these checks to make the result more than a single number: it becomes a controlled numerical experiment.